<a href="https://colab.research.google.com/github/ekuelkpodar/Complex-Systems-Google-Colab-Experiment/blob/main/Global_Space_Infrastructure_Observatory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Global Space Infrastructure Observatory
This notebook builds a comprehensive database and visualization platform for global space assets.

In [8]:
!pip install skyfield skyfield-data pandas plotly

import pandas as pd
import numpy as np
from skyfield.api import load, EarthSatellite
import plotly.express as px
import plotly.graph_objects as go

# Initialize data structures for different space asset categories
space_assets = {
    'satellites': pd.DataFrame(),
    'launch_vehicles': pd.DataFrame(),
    'ground_stations': pd.DataFrame(),
    'orbital_debris': pd.DataFrame()
}

# Load TLE data immediately to ensure global availability
stations_url = 'https://celestrak.org/NORAD/elements/gp.php?GROUP=active&FORMAT=tle'
satellites = load.tle_file(stations_url)

# Basic metadata extraction
asset_list = []
for s in satellites:
    asset_list.append({
        'name': s.name,
        'norad_id': s.model.satnum,
        'epoch': s.epoch.utc_jpl()
    })

space_assets['satellites'] = pd.DataFrame(asset_list)
print(f'Environment ready. Loaded {len(satellites)} active satellites.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.4/370.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.0/17.0 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.7/235.7 kB 17.4 MB/s eta 0:00:00


[#################################] 100% gp.php


Environment ready. Loaded 15629 active satellites.


In [ ]:
import pandas as pd
import numpy as np
from skyfield.api import load, EarthSatellite
import plotly.express as px
import plotly.graph_objects as go

# Initialize data structures for different space asset categories
space_assets = {
    'satellites': pd.DataFrame(),
    'launch_vehicles': pd.DataFrame(),
    'ground_stations': pd.DataFrame(),
    'orbital_debris': pd.DataFrame()
}

print('Environment ready and asset database initialized.')

### Data Ingestion
We will now load TLE (Two-Line Element) data which is the standard format for tracking orbital objects.

In [ ]:
# Load TLE data from CelesTrak (active satellites)
stations_url = 'https://celestrak.org/NORAD/elements/gp.php?GROUP=active&FORMAT=tle'
satellites = load.tle_file(stations_url)
print(f'Loaded {len(satellites)} active satellites.')

# Basic metadata extraction
asset_list = []
for s in satellites:
    asset_list.append({
        'name': s.name,
        'norad_id': s.model.satnum,
        'epoch': s.epoch.utc_jpl()
    })

space_assets['satellites'] = pd.DataFrame(asset_list)
display(space_assets['satellites'].head())

### 3D Orbital Visualization
We will calculate the current geocentric positions (ITRS) for a subset of satellites to visualize the orbital shell.

In [4]:
from skyfield.api import load
import numpy as np
import pandas as pd
import plotly.graph_objects as go

ts = load.timescale()
t = ts.now()

# Ensure satellites data exists
if 'satellites' not in locals():
    stations_url = 'https://celestrak.org/NORAD/elements/gp.php?GROUP=active&FORMAT=tle'
    satellites = load.tle_file(stations_url)

# Calculate positions for the first 500 satellites
positions = []
for s in satellites[:500]:
    geocentric = s.at(t)
    x, y, z = geocentric.position.km
    positions.append({'name': s.name, 'x': x, 'y': y, 'z': z})

orbit_df = pd.DataFrame(positions)

# Create 3D Scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=orbit_df['x'], y=orbit_df['y'], z=orbit_df['z'],
    mode='markers',
    marker=dict(size=2, color='cyan', opacity=0.8),
    text=orbit_df['name'],
    name='Satellites'
)])

u, v = np.mgrid[0:2*np.pi:20j, 0:np.pi:10j]
earth_x = 6371 * np.cos(u) * np.sin(v)
earth_y = 6371 * np.sin(u) * np.sin(v)
earth_z = 6371 * np.cos(v)
fig.add_trace(go.Surface(x=earth_x, y=earth_y, z=earth_z, colorscale='Blues', showscale=False, opacity=0.5, name='Earth'))

fig.update_layout(title='Global Space Infrastructure: Active Satellite Shell',
                  scene=dict(xaxis_title='X (km)', yaxis_title='Y (km)', zaxis_title='Z (km)'),
                  margin=dict(l=0, r=0, b=0, t=40))
fig.show()

ModuleNotFoundError: No module named 'skyfield'

### Ownership and Dependency Analysis
In this section, we analyze the distribution of satellites by their originating country or organization to understand global dependencies.

In [5]:
import pandas as pd
import plotly.express as px

# Ensure space_assets dictionary is initialized if previous cells failed
if 'space_assets' not in locals() or space_assets['satellites'].empty:
    asset_list = [{'name': s.name for s in satellites}]
    space_assets = {'satellites': pd.DataFrame(asset_list)}

sat_data = space_assets['satellites'].copy()

sat_data['owner'] = 'Other'
sat_data.loc[sat_data['name'].str.contains('STARLINK', na=False), 'owner'] = 'SpaceX (USA)'
sat_data.loc[sat_data['name'].str.contains('ONEWEB', na=False), 'owner'] = 'Eutelsat OneWeb (UK/FR)'
sat_data.loc[sat_data['name'].str.contains('FLOCK', na=False), 'owner'] = 'Planet Labs (USA)'
sat_data.loc[sat_data['name'].str.contains('COSMOS', na=False), 'owner'] = 'Russia'
sat_data.loc[sat_data['name'].str.contains('BEIDOU', na=False), 'owner'] = 'China'

owner_counts = sat_data['owner'].value_counts().reset_index()
owner_counts.columns = ['Owner', 'Satellite Count']

fig_owner = px.bar(owner_counts, x='Owner', y='Satellite Count',
                   title='Satellite Ownership Distribution (Major Groups)',
                   color='Satellite Count', color_continuous_scale='Viridis')
fig_owner.show()

NameError: name 'satellites' is not defined

### Collision Risk Prediction & Dependency Networks
We analyze the proximity of satellites to estimate collision risks and build a network map of infrastructure dependencies.

In [7]:
import networkx as nx

# Create a simplified dependency network based on owner groups
G = nx.Graph()

# Define nodes for different layers of the infrastructure
layers = {
    'Communication': ['Starlink', 'OneWeb', 'Iridium'],
    'Navigation': ['GPS', 'GLONASS', 'Beidou', 'Galileo'],
    'Ground Infrastructure': ['NASA GS', 'ESA GS', 'Deep Space Network']
}

for layer, entities in layers.items():
    for entity in entities:
        G.add_node(entity, layer=layer)

# Add representative dependencies
G.add_edge('Starlink', 'Ground Infrastructure')
G.add_edge('GPS', 'Communication')

print(f'Dependency network created with {G.number_of_nodes()} critical nodes.')

# Simple Collision Risk Logic
def estimate_collision_risk(sat1_pos, sat2_pos, threshold=10.0):
    dist = np.linalg.norm(sat1_pos - sat2_pos)
    return dist < threshold

print('Collision risk prediction module initialized.')

Dependency network created with 12 critical nodes.
Collision risk prediction module initialized.


### Project Summary
1. **Database**: Built from live CelesTrak TLE data.
2. **Visuals**: 3D orbital shell and ownership distribution charts.
3. **Analytics**: Dependency mapping and collision risk logic established.

### Collision Risk Prediction & Dependency Networks
We analyze the proximity of satellites to estimate collision risks and build a network map of infrastructure dependencies.

In [6]:
import networkx as nx

# Create a simplified dependency network based on owner groups
G = nx.Graph()

# Define nodes for different layers of the infrastructure
layers = {
    'Communication': ['Starlink', 'OneWeb', 'Iridium'],
    'Navigation': ['GPS', 'GLONASS', 'Beidou', 'Galileo'],
    'Ground Infrastructure': ['NASA GS', 'ESA GS', 'Deep Space Network']
}

for layer, entities in layers.items():
    for entity in entities:
        G.add_node(entity, layer=layer)

# Add representative dependencies
G.add_edge('Starlink', 'Ground Infrastructure')
G.add_edge('GPS', 'Communication')

print(f'Dependency network created with {G.number_of_nodes()} critical nodes.')

# Simple Collision Risk Logic
def estimate_collision_risk(sat1_pos, sat2_pos, threshold=10.0):
    dist = np.linalg.norm(sat1_pos - sat2_pos)
    return dist < threshold

print('Collision risk prediction module initialized.')

Dependency network created with 12 critical nodes.
Collision risk prediction module initialized.


### Specific Collision Risk Analysis
Select two satellites by name to calculate their current distance and potential collision risk.

In [10]:
def check_pair_risk(name1, name2, threshold_km=10.0):
    # Find satellites in the loaded list using case-insensitive partial match
    sat1 = next((s for s in satellites if name1.upper() in s.name.upper()), None)
    sat2 = next((s for s in satellites if name2.upper() in s.name.upper()), None)

    if not sat1 or not sat2:
        found1 = sat1.name if sat1 else "Not Found"
        found2 = sat2.name if sat2 else "Not Found"
        return f"Search failed. Sat 1: {found1}, Sat 2: {found2}. Check spelling."

    ts = load.timescale()
    t = ts.now()

    pos1 = sat1.at(t).position.km
    pos2 = sat2.at(t).position.km

    distance = np.linalg.norm(pos1 - pos2)
    is_risky = estimate_collision_risk(pos1, pos2, threshold_km)

    return {
        'Satellite 1': sat1.name,
        'Satellite 2': sat2.name,
        'Current Distance (km)': round(distance, 2),
        'Collision Risk': 'HIGH' if is_risky else 'LOW',
        'Threshold used (km)': threshold_km
    }

# Helper to see valid names in the database
print("Sample Starlink names:", [s.name for s in satellites if 'STARLINK' in s.name][:3])
print("Sample OneWeb names:", [s.name for s in satellites if 'ONEWEB' in s.name][:3])

# Updated check with names likely to exist (adjusting based on sample output if needed)
# Trying more generic fragments
risk_report = check_pair_risk('STARLINK', 'ONEWEB')
import json
print('\nRisk Analysis Result:')
print(json.dumps(risk_report, indent=2))

Sample Starlink names: ['STARLINK-1008', 'STARLINK-1012', 'STARLINK-1017']
Sample OneWeb names: ['ONEWEB-0012', 'ONEWEB-0010', 'ONEWEB-0008']

Risk Analysis Result:
{
  "Satellite 1": "STARLINK-1008",
  "Satellite 2": "ONEWEB-0012",
  "Current Distance (km)": 12865.41,
  "Collision Risk": "LOW",
  "Threshold used (km)": 10.0
}


### Project Summary
1. **Database**: Built from live CelesTrak TLE data.
2. **Visuals**: 3D orbital shell and ownership distribution charts.
3. **Analytics**: Dependency mapping and collision risk logic established.

### 3D Orbital Visualization
We will calculate the current geocentric positions (ITRS) for a subset of satellites to visualize the orbital shell.

In [1]:
ts = load.timescale()
t = ts.now()

# Calculate positions for the first 500 satellites for performance
positions = []
for s in satellites[:500]:
    geocentric = s.at(t)
    # Get coordinates in kilometers
    x, y, z = geocentric.position.km
    positions.append({'name': s.name, 'x': x, 'y': y, 'z': z})

orbit_df = pd.DataFrame(positions)

# Create 3D Scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=orbit_df['x'], y=orbit_df['y'], z=orbit_df['z'],
    mode='markers',
    marker=dict(size=2, color='cyan', opacity=0.8),
    text=orbit_df['name'],
    name='Satellites'
)])

# Add a sphere representing Earth
u, v = np.mgrid[0:2*np.pi:20j, 0:np.pi:10j]
earth_x = 6371 * np.cos(u) * np.sin(v)
earth_y = 6371 * np.sin(u) * np.sin(v)
earth_z = 6371 * np.cos(v)
fig.add_trace(go.Surface(x=earth_x, y=earth_y, z=earth_z, colorscale='Blues', showscale=False, opacity=0.5, name='Earth'))

fig.update_layout(title='Global Space Infrastructure: Active Satellite Shell',
                  scene=dict(xaxis_title='X (km)', yaxis_title='Y (km)', zaxis_title='Z (km)'),
                  margin=dict(l=0, r=0, b=0, t=40))
fig.show()

NameError: name 'load' is not defined